# 12 — Predict and evaluate the saved April–June test set

**Purpose:** use the existing published dashboard classifier to predict surge/no-surge on the saved test set, then compare with its actual next-week labels. Run All in the matching saved Python environment. There are **no new dates, raw downloads, protocol modes, model fitting or cutoff search**.

The test period was exposed during earlier development. These are **retrospective test-set results for the frozen model**, not an untouched independent final evaluation or live forecasts. The target remains an observed local median rise of at least **10% in the immediately following calendar week**, not an exact future RM price.

**Required locally:** the active run, existing deployment pin, matching comparison/study, fitted model, published export provenance and `experiments/holdout/05_modelling/test.parquet`. Missing or changed inputs stop execution; nothing is retrained or regenerated. Main notebooks 01–13 and the dashboard remain unchanged. Outputs are saved separately under this run’s `research/12_test_set_evaluation/` and displayed below.

In [ ]:
import ast
import hashlib
import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

REPOSITORY = Path.cwd().resolve()
if not (REPOSITORY / "pricecatcher_config.py").is_file():
    REPOSITORY = REPOSITORY.parent
if not (REPOSITORY / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open the FYP project or research folder before running this notebook.")

In [ ]:
RUN_DIRECTORY = None  # Normally leave unchanged; follows the existing active-run pointer.

## 1. Verify the frozen model and test inputs

Read the existing pin rather than selecting the latest study or creating a deployment. Check its study/comparison hashes, published model identity, Python/package versions and recorded test-file hash. Reuse only the saved model-input function from main 12; no other notebook cells run. The project settings module is not imported, avoiding its directory-creation side effects.

In [ ]:
def file_hash(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text())


def inside(root, relative):
    path = (Path(root) / relative).resolve()
    if not path.is_relative_to(Path(root).resolve()):
        raise ValueError('A saved path escapes its expected folder.')
    return path


def calculations(run):
    """Reuse named read-only helpers without importing settings or executing notebooks."""
    namespace = {'Path': Path, 'hashlib': hashlib, 'json': json, 'np': np, 'pd': pd,
                 '__file__': str(REPOSITORY / 'pricecatcher_config.py'),
                 'PROJECT_ROOT': run, 'DATA_HOME': REPOSITORY / 'data'}
    tree = ast.parse((REPOSITORY / 'pricecatcher_config.py').read_text())
    mapping = next(node for node in tree.body if isinstance(node, ast.Assign)
                   and any(isinstance(t, ast.Name) and t.id == 'RESEARCH_NOTEBOOK_NAMES' for t in node.targets))
    names = {'installed_package_version', 'resolve_input', 'notebook_path', 'input_reference'}
    nodes = [mapping] + [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name in names]
    exec(compile(ast.Module(body=nodes, type_ignores=[]), 'read-only helpers', 'exec'), namespace)
    notebook = read_json(REPOSITORY / '12_shap_and_dashboard_inputs.ipynb')
    functions = [node for cell in notebook['cells'] if cell['cell_type'] == 'code'
                 for node in ast.parse(''.join(cell['source'])).body
                 if isinstance(node, ast.FunctionDef) and node.name == 'model_inputs']
    if len(functions) != 1:
        raise ValueError('Expected one saved model-input calculation in main Notebook 12.')
    exec(compile(ast.Module(body=functions, type_ignores=[]), 'model input preparation only', 'exec'), namespace)
    return namespace


def verify_inputs(run):
    """Require the existing published pin and its exact test-set identity; never create a pin."""
    protected = {}
    def remember(path, expected=None):
        path = Path(path).resolve()
        actual = file_hash(path)
        if expected is not None and actual != expected:
            raise ValueError(f'Frozen file changed: {path}. Restore the matching artifact; do not rewrite hashes.')
        protected[str(path)] = actual
        return path
    pin_path = run / '11_deployment/deployment_selection.json'
    if not pin_path.is_file():
        raise FileNotFoundError('Restore the existing deployment pin and matching run. No model or deployment is created here.')
    pin = read_json(remember(pin_path))
    study_path = remember(inside(run, pin['study_manifest']), pin['study_manifest_sha256'])
    comparison_path = remember(inside(run, pin['comparison_manifest']), pin['comparison_manifest_sha256'])
    study, comparison = read_json(study_path), read_json(comparison_path)
    declared = comparison['studies'][pin['candidate']]
    identity = {'study_sha256': pin['study_manifest_sha256'], 'cutoff': pin['score_cutoff']}
    if (study.get('status') != 'completed' or comparison.get('status') != 'completed'
            or comparison['candidate_under_fold_rule'] != pin['candidate']
            or inside(run, declared['folder']) != study_path.parent
            or declared['manifest_sha256'] != pin['study_manifest_sha256']
            or hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16] != pin['deployment_id']):
        raise ValueError('Pin, study and comparison identify different model selections.')
    selection = read_json(remember(study_path.parent / 'selection.json'))
    if selection != study['selection'] or selection['score_cutoff'] != pin['score_cutoff']:
        raise ValueError('Frozen selection or cutoff changed.')
    cutoff = float(selection['score_cutoff'])
    if not np.isfinite(cutoff) or not 0 <= cutoff <= 1 or selection['surge_threshold_pct'] != 10:
        raise ValueError('Expected the frozen score cutoff and existing 10% target.')
    for name, expected in comparison['output_sha256'].items():
        remember(inside(comparison_path.parent, name), expected)
    for name, expected in study['output_sha256'].items():
        remember(inside(study_path.parent, name), expected)
    for name, expected in study['input_sha256'].items():
        remember(inside(run, name), expected)
    helpers = calculations(run)
    for name, expected in study.get('additional_input_sha256', {}).items():
        remember(helpers['resolve_input'](name), expected)
    if sys.version_info[:2] != tuple(map(int, study.get('python_version', '3.12').split('.')[:2])):
        raise ValueError('Use the Python major/minor version recorded by this study.')
    for package, expected in study['package_versions'].items():
        if helpers['installed_package_version'](package) != expected:
            raise ValueError(f'Restore the saved {package} version before predicting.')
    prepared = run / 'experiments/holdout/05_modelling'
    settings_path = prepared / 'experiment_settings.json'
    settings = read_json(remember(settings_path, study['input_sha256'][settings_path.relative_to(run).as_posix()]))
    if (settings['train_end'], settings['validation_end'], settings['test_end']) != ('2025-12-31', '2026-03-31', '2026-06-30'):
        raise ValueError('This notebook evaluates only the original configured April–June 2026 test period.')
    if settings['surge_threshold_pct'] != 10 or selection['training_end'] != settings['train_end']:
        raise ValueError('Target or final training boundary differs from the frozen settings.')
    features = settings['feature_columns']
    if (features != selection['feature_columns'] or len(features) != len(set(features))
            or sorted(features) != sorted(settings['numeric_features'] + settings['categorical_features'])
            or any(name.startswith('next_') or name in {'surge_label', 'split'} for name in features)):
        raise ValueError('Frozen feature definitions differ or contain outcomes.')
    dashboard = run / '08_dashboard'
    pointer = dashboard / 'active_deployment.json'
    if pointer.is_file():
        record = read_json(remember(pointer))
        dashboard = inside(dashboard, record['directory'])
        if not dashboard.is_relative_to((run / '08_dashboard/deployments').resolve()):
            raise ValueError('Invalid dashboard deployment path.')
    entry = read_json(remember(dashboard / 'dashboard_manifest.json'))
    published = entry['forecast_model']
    model = remember(inside(run, published['model_path']), published['sha256'])
    if (published.get('deployment_id') != pin['deployment_id'] or published['score_cutoff'] != cutoff
            or model.parent != study_path.parent or study['output_sha256'].get(model.name) != published['sha256']):
        raise ValueError('The published dashboard checkpoint differs from the deployment pin.')
    cache = inside(dashboard, entry['cache_dir'])
    manifest = read_json(remember(cache / 'explanation_manifest.json', entry['cache_manifest_sha256']))
    if manifest['forecast_model'] != published:
        raise ValueError('Dashboard entry and cache describe different models.')
    test_path = prepared / 'test.parquet'
    expected_test = manifest['upstream_sha256'].get('prepared/holdout/test.parquet')
    if expected_test is None:
        raise ValueError('The original dashboard export does not record the test-set hash. Restore matching provenance; nothing is regenerated.')
    remember(test_path, expected_test)
    for path in [dashboard / 'refresh_manifest.json', run.parent.parent / 'active_run.txt']:
        if path.is_file():
            remember(path)
    for path in sorted((run / '13_registered_future_evaluation').glob('*/registration.json')):
        remember(path)
    for path in [REPOSITORY / 'pricecatcher_config.py', REPOSITORY / '12_shap_and_dashboard_inputs.ipynb']:
        remember(path)
    return pin, selection, settings, model, test_path, protected, helpers

## 2. Keep the original test population and cutoff

Check complete calendar-week horizons, split boundaries, unique series-weeks and observed binary labels. Follow main 05 exactly: the **current week end** must follow 31 March, and the **next-week outcome end** must be on or before 30 June. This avoids accidentally dropping a valid boundary week by filtering on its Monday.

Predict using the fitted pipeline’s existing preprocessing and classifier. Do not call `fit`, choose a new threshold, drop inconvenient groups or substitute missing outcomes with zero. Blank metrics indicate an undefined quantity (for example, recall with no observed surges); single-class AP is not assessed.

In [ ]:
def validate_test(frame, settings):
    keys = ['item_code', 'state', 'district', 'premise_type', 'week_start']
    required = list(dict.fromkeys(settings['feature_columns'] + keys +
        ['item', 'unit', 'project_group', 'next_week_end', 'surge_label']))
    if frame.empty or set(required) - set(frame.columns):
        raise ValueError('Saved test rows are empty or missing required columns.')
    frame = frame[required].copy()
    for name in ['week_start', 'next_week_end']:
        frame[name] = pd.to_datetime(frame[name], errors='raise')
    if (frame[keys + ['next_week_end', 'project_group']].isna().any().any()
            or frame.duplicated(keys).any() or not frame.week_start.dt.day_of_week.eq(0).all()
            or not frame.next_week_end.eq(frame.week_start + pd.Timedelta(days=13)).all()):
        raise ValueError('Invalid dates, missing identifiers or duplicate local-series weeks.')
    # Match main 05 exactly: check CURRENT week end, not its Monday, against validation end.
    if (not (frame.week_start + pd.Timedelta(days=6)).gt(pd.Timestamp(settings['validation_end'])).all()
            or not frame.next_week_end.le(pd.Timestamp(settings['test_end'])).all()):
        raise ValueError('Test outcomes cross the frozen split boundaries.')
    if frame.surge_label.isna().any() or not frame.surge_label.isin([0, 1]).all():
        raise ValueError('Test outcomes must be observed binary surge labels; missing outcomes are not negatives.')
    if not np.isfinite(frame[settings['numeric_features']].fillna(0).to_numpy(dtype=float)).all():
        raise ValueError('Non-finite test features found. Missing features must retain the saved training-fitted preprocessing.')
    frame['surge_label'] = frame.surge_label.astype(int)
    frame['outcome_week'] = frame.week_start + pd.Timedelta(days=7)
    return frame


def metrics(frame):
    y, predicted = frame.surge_label.to_numpy(dtype=int), frame.predicted_surge.to_numpy(dtype=bool)
    tp = int((predicted & (y == 1)).sum()); fp = int((predicted & (y == 0)).sum())
    fn = int((~predicted & (y == 1)).sum()); tn = int((~predicted & (y == 0)).sum())
    return {'rows': len(frame), 'observed_surges': int(y.sum()), 'alerts': int(predicted.sum()),
        'average_precision': float(average_precision_score(y, frame.model_score)) if len(np.unique(y)) == 2 else np.nan,
        'precision': tp / (tp + fp) if tp + fp else np.nan,
        'recall': tp / (tp + fn) if tp + fn else np.nan,
        'f1': 2 * tp / (2 * tp + fp + fn) if y.sum() else np.nan,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}


def check_unchanged(protected, resolve=None):
    for name, expected in protected.items():
        path = resolve(name) if resolve is not None else Path(name)
        if file_hash(path) != expected:
            raise ValueError(f'Protected input changed: {name}. No completed result can be published.')

## 3. Save one separate evaluation

A fingerprint ties the results to the model, test-file bytes, cutoff and evaluation code. Repetition verifies and reuses completed results; it is not another independent test. A partial output folder is never overwritten or deleted automatically. Protected input hashes are rechecked before publication.

Files: individual predictions in Parquet/CSV, overall metrics, a confusion matrix, weekly metrics, food-group metrics and an evaluation manifest. The normal dashboard exports and registrations are not changed.

In [ ]:
def evaluate_test(run):
    pin, selection, settings, model, test_path, protected, helpers = verify_inputs(run)
    notebook = read_json(REPOSITORY / 'research/12_test_set_evaluation.ipynb')
    code = '\n'.join(''.join(cell['source']) for cell in notebook['cells']
                     if cell['cell_type'] == 'code' and 'configuration' not in cell.get('metadata', {}).get('tags', []))
    identity = {'deployment_id': pin['deployment_id'], 'candidate': pin['candidate'],
        'model_sha256': file_hash(model), 'test_sha256': file_hash(test_path),
        'score_cutoff': float(selection['score_cutoff']), 'calculation_sha256': hashlib.sha256(code.encode()).hexdigest(),
        'evaluation_status': 'retrospective test-set evaluation; April–June was previously exposed during development'}
    evaluation_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
    output = inside(run, 'research/12_test_set_evaluation') / evaluation_id
    saved_manifest = output / 'evaluation_manifest.json'
    if saved_manifest.is_file():
        saved = read_json(saved_manifest)
        if saved['identity'] != identity:
            raise ValueError('Existing evaluation belongs to different inputs.')
        check_unchanged(saved['input_sha256'], helpers['resolve_input'])
        for name, expected in saved['output_sha256'].items():
            if file_hash(inside(output, name)) != expected:
                raise ValueError('Saved evaluation output changed. Restore it; nothing is overwritten.')
        print('Verified existing test evaluation; reusing results, not another independent test.')
        return output
    if output.exists():
        raise ValueError('Partial evaluation folder exists. Preserve and inspect it; no automatic deletion or overwrite.')
    frame = validate_test(pd.read_parquet(test_path), settings)
    pipeline = joblib.load(model)
    if list(pipeline.classes_) != [0, 1]:
        raise ValueError('Expected the saved binary 0/1 classifier.')
    scores = pipeline.predict_proba(helpers['model_inputs'](frame, settings))[:, 1]
    if len(scores) != len(frame) or not np.isfinite(scores).all() or not ((scores >= 0) & (scores <= 1)).all():
        raise ValueError('The frozen model returned invalid test scores.')
    frame['model_score'] = scores
    frame['predicted_surge'] = scores >= identity['score_cutoff']
    frame['outcome'] = np.select([frame.predicted_surge & frame.surge_label.eq(1),
        frame.predicted_surge & frame.surge_label.eq(0), ~frame.predicted_surge & frame.surge_label.eq(1)],
        ['correct_alert', 'false_alert', 'missed_surge'], default='correct_no_alert')
    overall = pd.DataFrame([metrics(frame)])
    groups = pd.DataFrame([{'project_group': group, **metrics(part)} for group, part in frame.groupby('project_group', sort=True)])
    weekly = pd.DataFrame([{'outcome_week': str(week.date()), **metrics(part)} for week, part in frame.groupby('outcome_week', sort=True)])
    m = overall.iloc[0]
    confusion = pd.DataFrame([[int(m.tn), int(m.fp)], [int(m.fn), int(m.tp)]],
        index=['Actual no surge', 'Actual surge'], columns=['Predicted no surge', 'Predicted surge'])
    check_unchanged(protected)
    output.mkdir(parents=True, exist_ok=False)
    columns = ['item_code', 'item', 'unit', 'project_group', 'state', 'district', 'premise_type',
               'week_start', 'outcome_week', 'next_week_end', 'surge_label', 'model_score', 'predicted_surge', 'outcome']
    frame[columns].to_parquet(output / 'test_predictions.parquet', index=False)
    frame[columns].to_csv(output / 'test_predictions.csv', index=False)
    for filename, table in [('overall_metrics.csv', overall), ('food_group_metrics.csv', groups), ('weekly_metrics.csv', weekly)]:
        table.to_csv(output / filename, index=False)
    confusion.to_csv(output / 'confusion_matrix.csv', index_label='Actual outcome')
    check_unchanged(protected)
    references = {}
    for name, expected in protected.items():
        try:
            reference = helpers['input_reference'](name)
        except ValueError:
            reference = name  # Preserve an explicitly recorded external legacy input.
        references[reference] = expected
    saved = {'status': 'completed', 'identity': identity, 'python_version': sys.version.split()[0],
        'completed_at': pd.Timestamp.now(tz='Asia/Kuala_Lumpur').isoformat(),
        'configured_period': {'after': settings['validation_end'], 'through': settings['test_end']},
        'actual_outcome_weeks': {'first': str(frame.outcome_week.min().date()), 'last': str(frame.outcome_week.max().date()),
                                 'weeks': int(frame.outcome_week.nunique())},
        'test_rows': len(frame), 'input_sha256': references,
        'output_sha256': {path.name: file_hash(path) for path in sorted(output.iterdir()) if path.is_file()},
        'training_performed': False, 'threshold_selected_on_test': False, 'dashboard_modified': False,
        'limitations': ['previous exposure means this is not an untouched independent final test',
                        'metrics apply to supported saved test rows, not every shop or district',
                        'weeks and local series are dependent; no independent-row uncertainty claim',
                        'single-class AP and undefined precision/recall/F1 are blank',
                        'scores are uncalibrated, not verified probabilities']}
    with saved_manifest.open('x') as handle:
        json.dump(saved, handle, indent=2)
    return output

## 4. Run predictions and view the outputs

With the matching artifacts on your Mac, **Restart Kernel and Run All**. The output folder is printed below. No separate freeze/evaluate configuration is required. `data/` is excluded from Git, so the result files stay local unless you explicitly share them.

The cloud checkout has no real saved data/models. This notebook has only been checked statically and with synthetic fixtures; no real test performance is claimed here.

In [ ]:
if RUN_DIRECTORY is not None:
    RUN = Path(RUN_DIRECTORY).expanduser().resolve()
elif os.environ.get("PRICECATCHER_DATA_DIR"):
    RUN = Path(os.environ["PRICECATCHER_DATA_DIR"]).expanduser().resolve()
else:
    pointer = REPOSITORY / "data" / "active_run.txt"
    if not pointer.is_file():
        raise FileNotFoundError("Restore data/active_run.txt and its matching saved run, or set RUN_DIRECTORY. No outputs were created.")
    relative = Path(pointer.read_text().strip())
    if relative.is_absolute() or relative.parts[:1] != ("runs",) or ".." in relative.parts:
        raise ValueError("Invalid active-run pointer.")
    RUN = inside(REPOSITORY / "data", relative)

OUTPUT = evaluate_test(RUN)
record = read_json(OUTPUT / "evaluation_manifest.json")
print("Model:", record["identity"]["candidate"])
print("Frozen cutoff:", record["identity"]["score_cutoff"])
print("Evidence status:", record["identity"]["evaluation_status"])
print("Test rows:", record["test_rows"], "| outcome weeks:", record["actual_outcome_weeks"])
print("Results saved to:", OUTPUT)
print("No retraining, cutoff selection, dashboard updates or registration changes.")
for filename in ["overall_metrics.csv", "confusion_matrix.csv", "food_group_metrics.csv", "weekly_metrics.csv"]:
    print(filename)
    display(pd.read_csv(OUTPUT / filename))
print("First 10 predictions (all rows are saved in test_predictions.csv/parquet):")
display(pd.read_parquet(OUTPUT / "test_predictions.parquet").head(10))

## Use in the report

- **AP:** ranking quality; not classification accuracy or a calibrated probability.
- **Precision:** how many alerts corresponded to observed surges. **Recall:** how many observed surges were alerted. **F1:** balance of precision and recall at the already frozen cutoff.
- In the confusion matrix, rows are **actual outcomes** and columns are **predictions**. False alerts and missed surges should both be reported.
- Include every food group and its positive count. Sparse/zero-positive groups do not support strong reliability claims.
- State the actual outcome-week range and eligible row counts printed above. Supported sampled medians are not every shop’s price.
- Disclose prior April–June exposure. Do not tune against these results and relabel the same period as independent. Preserve weaker results as well as stronger ones.